# CellLM Experiment 0 — GPU ladder

Runs the complete eight-rung, three-seed capacity gate. Completed seed checkpoints are written to the submitter's mounted output directory so they survive the job.

In [ ]:
%pip install git+https://github.com/tomieiro/libPyCelNN.git
%pip install -e . --no-deps

In [ ]:
from pathlib import Path
import importlib
import shutil
import sys

root = Path('/workspace')
outputs = root / 'outputs'
output_checkpoints = outputs / 'checkpoints'
output_checkpoints.mkdir(parents=True, exist_ok=True)

# Checkpoints copied back into the project before a resubmission are staged
# into the mounted output directory and skipped by the ladder runner.
local_checkpoints = root / 'checkpoints'
if local_checkpoints.exists():
    for checkpoint in local_checkpoints.glob('*.pt'):
        destination = output_checkpoints / checkpoint.name
        if not destination.exists():
            shutil.copy2(checkpoint, destination)

data_dir = root / 'data'
text8 = data_dir / 'text8'
if not text8.exists():
    shutil.unpack_archive(data_dir / 'text8.zip', data_dir)

torch = importlib.import_module('torch')
assert torch.cuda.is_available(), 'the submitted job did not receive a GPU'
print('GPU:', torch.cuda.get_device_name(0), flush=True)

command = [
    sys.executable, '-m', 'celllm.ladder',
    '--data', str(text8),
    '--steps', '20000',
    '--device', 'cuda',
    '--checkpoint-dir', str(output_checkpoints),
    '--parallel-seeds', '3',
    '--out', str(outputs / 'experiment-0.json'),
]
print(' '.join(command), flush=True)
__import__('subprocess').run(command, check=True)